# Urban Green Cover and Population Growth
**Problem Statement:** Examine how urban green-space indicates changes alongside population growth and urban expansion

### Data Sources
- **Green Cover:** City NDVI(a satellite greenness index,higher = greener) fro 2010, 2015, 2020, 2021 from the city greenspace dataset.
- **Population and Density:** UN DESA *World Urbanization Prospects 2025*(Files 21 and 34).
- **Land Area:** calculated by me as population/density(the UN land-ares files was not used).

In [2]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

os.makedirs("charts", exist_ok=True)
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 1: Collect population, land-use and green-space indicators for comparable places and time periods.
I have loaded the 4 sheet of the prepared dataset and what is inside

In [3]:
file_name = "Urban_Green_Cover_Final_Dataset.xlsx"
wide = pd.read_excel(file_name, sheet_name="City_Summary_Wide")
long = pd.read_excel(file_name, sheet_name="City_Year_Long")
unmatched = pd.read_excel(file_name, sheet_name="Unmatched_Cities")
dictionary = pd.read_excel(file_name, sheet_name="Data_Dictionary")

print("Wide table shape:", wide.shape[0])
print("Long table shape:", long.shape[0])
print("Unmatched cities shape:", unmatched.shape[0])
print("Dictionary shape:", dictionary.shape[0])

Wide table shape: 1038
Long table shape: 4152
Unmatched cities shape: 85
Dictionary shape: 14


In [4]:
wide.head(3)

,City,Country,Major_Geo_Region,HDI_level,Climate_region,WHO_region,ISO3_Code,City_Code,WUP_City_Name,Match_Type,...,WUP_Land_Area_km2_2020,WUP_Land_Area_km2_2021,NDVI_change_2010_2020,Pop_growth_pct_2010_2020,Area_growth_pct_2010_2020,NDVI_change_2015_2020,Pop_growth_pct_2015_2020,Area_growth_pct_2015_2020,Longitude,Latitude
0,Aba,Nigeria,Africa,Low,Tropical,Africa,NGA,7785.0,Aba,exact_name,...,184.999881,185.799877,0.059070,32.284117,28.472145,-0.076121,11.673534,3.351872,7.352215,5.102898
1,Abbottabad,Pakistan,Asia,Medium,Temperate,Eastern Mediterranean,PAK,7045.0,Abbottabad,exact_name,...,107.000052,107.800064,0.050275,9.421621,13.829890,-0.005598,0.660094,4.902022,73.228832,34.177822
2,Abeokuta,Nigeria,Africa,Low,Tropical,Africa,NGA,1570.0,Abeokuta,exact_name,...,169.000039,170.799917,0.082825,16.551963,8.333269,-0.009434,7.653565,3.681046,3.342968,7.161728


In [5]:
long.head(3)

,City,Country,Major_Geo_Region,HDI_level,Climate_region,WHO_region,ISO3_Code,City_Code,WUP_City_Name,Match_Type,Year,NDVI_annual_avg,NDVI_peak,NDVI_annual_pop_weighted,Greenness_Level,WUP_Population_thousands,WUP_Land_Area_km2,WUP_Density_per_km2,Longitude,Latitude
0,Herat,Afghanistan,Asia,Low,Arid,Eastern Mediterranean,AFG,742.0,Herat,exact_name,2010,0.164070,0.220683,0.187444,Very Low,668.974,70.999993,9422.17,62.205237,34.349858
1,Herat,Afghanistan,Asia,Low,Arid,Eastern Mediterranean,AFG,742.0,Herat,exact_name,2015,0.176511,0.239405,0.196993,Very Low,878.077,75.000021,11707.69,62.205237,34.349858
2,Herat,Afghanistan,Asia,Low,Arid,Eastern Mediterranean,AFG,742.0,Herat,exact_name,2020,0.164452,0.205323,0.183483,Very Low,1181.815,81.999989,14412.38,62.205237,34.349858


In [6]:
wide.dtypes

City                                 str
Country                              str
Major_Geo_Region                     str
HDI_level                            str
Climate_region                       str
WHO_region                           str
ISO3_Code                            str
City_Code                        float64
WUP_City_Name                        str
Match_Type                           str
NDVI_annual_avg_2010             float64
NDVI_annual_avg_2015             float64
NDVI_annual_avg_2020             float64
NDVI_annual_avg_2021             float64
WUP_Population_thousands_2010    float64
WUP_Population_thousands_2015    float64
WUP_Population_thousands_2020    float64
WUP_Population_thousands_2021    float64
WUP_Land_Area_km2_2010           float64
WUP_Land_Area_km2_2015           float64
WUP_Land_Area_km2_2020           float64
WUP_Land_Area_km2_2021           float64
NDVI_change_2010_2020            float64
Pop_growth_pct_2010_2020         float64
Area_growth_pct_

In [7]:
indicators =pd.DataFrame({
    "Indicator":["Green cover(NDVI)", "Population", "Population density","Land area"],
    "Unit":["index(0 to 1,higher = greener)", "thousands of people", "people per sq.km", "sq.km"],
    "Source":["City greenspace dataset (satellite)","UN World Urbanization Prospects 2025 (File 21)","UN World Urbanization Prospects 2025 (File 34)","Derived population / density"],
    "Years":["2010,2015,2020,2021"] * 4
})
indicators

,Indicator,Unit,Source,Years
0,Green cover(NDVI),"index(0 to 1,higher = greener)",City greenspace dataset (satellite),"2010,2015,2020,2021"
1,Population,thousands of people,UN World Urbanization Prospects 2025 (File 21),"2010,2015,2020,2021"
2,Population density,people per sq.km,UN World Urbanization Prospects 2025 (File 34),"2010,2015,2020,2021"
3,Land area,sq.km,Derived population / density,"2010,2015,2020,2021"


**Question 1 summary:** the data has green cover (NDVI),population and land -use information (land  area and density) for 1038 cities in 172 countries, for the same 4 year (2010,2015, 2020, 2021) Land use is represented by the city's land area and population density;there is no separate land -cover classification.

## 2. Align geographical boundaries, years, units and location labels
We check that all cities share the same years,units and labels, then keep only the cities that were successfully match to a un city (these have the population data)

In [8]:
print("Years in the long table:", sorted(long['Year'].unique()))
print("Rows per Year:")
print(long["Year"].value_counts().sort_index())

Years in the long table: [np.int64(2010), np.int64(2015), np.int64(2020), np.int64(2021)]
Rows per Year:
Year
2010    1038
2015    1038
2020    1038
2021    1038
Name: count, dtype: int64


In [9]:
wide["City"]= wide["City"].str.strip()
wide["Country"]= wide["Country"].str.strip()
long["City"]= long["City"].str.strip()
long["Country"]= long["Country"].str.strip()
print("Duplicate city + country pairs in the wide table :",wide .duplicated(["City","Country"]).sum())
print("Number of  countries :",wide["Country"].nunique())

Duplicate city + country pairs in the wide table : 0
Number of  countries : 172


In [10]:
print("units are consistent: NDVI = index, population =thousands,area=sq.km,")
wide["Match_Type"].value_counts()

units are consistent: NDVI = index, population =thousands,area=sq.km,


Match_Type
exact_name                       900
unmatched                         85
name_ambiguous_largest_chosen     47
manual_alias                       6
Name: count, dtype: int64

In [11]:
matched=wide[wide["Match_Type"]!="unmatched"].copy()
match_long=long[long["Match_Type"]!="unmatched"].copy()
print("city in the full dataset:",len(wide))
print("city in the matched dataset:",len(matched))
print("Cities left out of the matched dataset:",len(wide)-len(matched))


city in the full dataset: 1038
city in the matched dataset: 953
Cities left out of the matched dataset: 85


In [12]:
needed =["NDVI_change_2015_2020", "Pop_growth_pct_2015_2020","Area_growth_pct_2015_2020"]
analysis=matched.dropna(subset=needed).copy()
keys=analysis[["City","Country"]]
analysis_long = match_long.merge(keys, on=["City", "Country"])
print("Cities in the analysis dataset:",len(analysis))
print()
print("Cities per region:")
print(analysis["Major_Geo_Region"].value_counts())


Cities in the analysis dataset: 948

Cities per region:
Major_Geo_Region
Asia                               507
Africa                             142
Europe                             123
Latin America and the Caribbean    116
Northern America                    51
Oceania                              9
Name: count, dtype: int64
